# LoRA Fine-Tuning (Rank=16) — Train on 1,000 Examples, Evaluate on 1,000 Examples

**Project:** LoRA Fine-Tuning for Text Summarization  
**Model:** Qwen2.5-7B  
**Dataset:** CNN/DailyMail (version 3.0.0)  
**Purpose:** Fine-tune Qwen2.5-7B using LoRA (rank=16) on 1,000 training examples, then evaluate on 1,000 test examples. This is the small-scale proof-of-concept experiment — useful for validating the training pipeline before scaling up to 287k examples.

---

### What is LoRA?
LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method. Instead of updating all 7 billion parameters of the model, LoRA introduces small trainable matrices (adapters) alongside the frozen pretrained weights. Only these adapters are trained, drastically reducing memory and compute requirements.

**Key LoRA parameters used here:**
- `lora_rank = 16` — the rank of the adapter matrices. Lower rank = fewer parameters = less risk of overfitting
- `lora_alpha = 512` — scaling factor. The effective scaling is `alpha / rank = 512 / 16 = 32`, kept constant across experiments
- `lora_dropout = 0.05` — dropout applied to LoRA layers for regularization
- `lora_target` — which weight matrices to apply LoRA to (all attention and MLP projection layers)

---

### What is LLamaFactory?
LLamaFactory is an open-source framework that simplifies fine-tuning of large language models. It handles the training loop, logging, checkpointing, and LoRA integration. We configure it via a YAML file and launch training with a single CLI command.

---

### Notebook Flow
1. Install libraries
2. Check GPU
3. Authenticate with HuggingFace
4. Load and format 1,000 training examples
5. Create LLamaFactory config files
6. Train LoRA rank=16 model
7. Load the fine-tuned model
8. Test on a single example
9. Evaluate on 1,000 test examples
10. Calculate BLEU and ROUGE scores
11. Save results

## Step 1 — Install Required Libraries
We install all dependencies including LLamaFactory, which is installed directly from its GitHub repository.

- `transformers` — load and run Qwen
- `peft` — Parameter-Efficient Fine-Tuning library, used to load LoRA adapters after training
- `datasets` — load CNN/DailyMail
- `accelerate` — efficient GPU training
- `bitsandbytes` — quantization support used by LLamaFactory
- `evaluate`, `rouge-score`, `nltk` — evaluation metrics
- `LLamaFactory` — the training framework

In [ ]:
!pip install transformers peft datasets accelerate bitsandbytes rouge-score nltk evaluate -q
!pip install git+https://github.com/hiyouga/LLaMA-Factory.git -q
print("All libraries installed successfully!")

## Step 2 — Check GPU Availability
We verify that a GPU is available via CUDA.

**Note on GPU memory:**  
Qwen2.5-7B is a 7-billion parameter model and requires a GPU with sufficient VRAM to run. The more VRAM available, the larger the batch size you can use during training and evaluation.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"   GPU is available!")
    print(f"   GPU Name     : {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   No GPU found. Please enable a GPU in your runtime settings.")
    print("   Qwen2.5-7B requires a GPU to run in reasonable time.")

## Step 3 — Authenticate with HuggingFace
Qwen2.5-7B is a publicly available model — unlike Llama, **no license acceptance is required**. You simply need a HuggingFace account and access token to download it.

1. Create a HuggingFace account at: https://huggingface.co/join
2. Generate an access token at: https://huggingface.co/settings/tokens

Paste your token when prompted below.

In [ ]:
from huggingface_hub import login

# This will prompt you to enter your HuggingFace access token
# Get your token from: https://huggingface.co/settings/tokens
# Note: Qwen2.5-7B is publicly available — no license acceptance needed
login()
print("Logged in to HuggingFace!")

## Step 4 — Load and Format Training Data (1,000 examples)

We load 1,000 examples from the CNN/DailyMail **training** split and format them for LLamaFactory.

**LLamaFactory's expected format:**  
Each training example must be a JSON object with three fields:
- `instruction` — the task description (same for all examples)
- `input` — the article to summarize
- `output` — the reference summary the model should learn to generate

**What is `dataset_info.json`?**  
LLamaFactory requires a registry file called `dataset_info.json` that maps a dataset name (used in the YAML config) to the actual file and its column structure. Without this file, LLamaFactory cannot find or read the training data.

In [ ]:
import json
import os
from datasets import load_dataset

print("Loading 1,000 training examples from CNN/DailyMail...")

# Load 1,000 examples from the training split
# split="train[:1000]" is HuggingFace slice notation — loads only the first 1,000 examples
train_dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:1000]")

print(f"   Loaded {len(train_dataset)} training examples")

# Format each example into LLamaFactory's required JSON structure
# instruction: the task description (constant across all examples)
# input:       the article (the model's input at training time)
# output:      the reference summary (what the model is trained to generate)
formatted_data = []
for example in train_dataset:
    formatted_data.append({
        "instruction": "Summarize the following news article.",
        "input":       example['article'],
        "output":      example['highlights']
    })

# Create a data/ directory to store training files
# LLamaFactory expects the dataset and dataset_info.json to be in the same folder
os.makedirs('data', exist_ok=True)

# Save the formatted training data
with open('data/train_data_1k.json', 'w') as f:
    json.dump(formatted_data, f, indent=2)

# Create dataset_info.json — this tells LLamaFactory how to read the data file
# 'train_data_1k' is the name we will reference in the YAML config
# The 'columns' section maps LLamaFactory's internal field names to our JSON keys
dataset_info = {
    "train_data_1k": {
        "file_name": "train_data_1k.json",
        "columns": {
            "prompt":   "instruction",
            "query":    "input",
            "response": "output"
        }
    }
}

with open('data/dataset_info.json', 'w') as f:
    json.dump(dataset_info, f, indent=2)

print(f"\n   Saved {len(formatted_data)} formatted examples to data/train_data_1k.json")
print(f"   Dataset registry saved to data/dataset_info.json")
print(f"\n--- Sample formatted example ---")
print(f"Instruction : {formatted_data[0]['instruction']}")
print(f"Input       : {formatted_data[0]['input'][:150]}...")
print(f"Output      : {formatted_data[0]['output']}")

## Step 5 — Create the LLamaFactory Training Config (YAML)

LLamaFactory is configured via a YAML file that specifies the model, dataset, LoRA settings, and all training hyperparameters. We create this file programmatically so it is fully reproducible.

**Key parameters explained:**

| Parameter | Value | Explanation |
|-----------|-------|-------------|
| `finetuning_type` | `lora` | Use LoRA adapters instead of full fine-tuning |
| `lora_rank` | `16` | Rank of the adapter matrices — controls number of trainable parameters |
| `lora_alpha` | `512` | Scaling factor. Effective scale = alpha/rank = 512/16 = 32 |
| `lora_dropout` | `0.05` | Dropout on LoRA layers to prevent overfitting |
| `lora_target` | `all` | Apply LoRA to all attention and MLP projection layers |
| `cutoff_len` | `512` | Maximum token length for input sequences |
| `per_device_train_batch_size` | `4` | Examples per GPU per step — reduce if out of memory |
| `gradient_accumulation_steps` | `4` | Accumulate gradients over 4 steps → effective batch size = 16 |
| `learning_rate` | `3e-4` | Standard learning rate for LoRA fine-tuning |
| `lr_scheduler_type` | `cosine` | Learning rate decays following a cosine curve |
| `warmup_ratio` | `0.05` | Linearly warm up learning rate for first 5% of steps |
| `num_train_epochs` | `3` | Train for 3 full passes over the data |
| `bf16` | `true` | Use bfloat16 precision — recommended for Qwen2.5 |
| `val_size` | `0.1` | Hold out 10% of training data for validation |
| `template` | `qwen` | Use Qwen's built-in prompt template |

In [ ]:
# Define the training configuration as a YAML string
config_r16_1k = """
### Model
model_name_or_path: Qwen/Qwen2.5-7B

### Method
stage: sft
do_train: true
finetuning_type: lora
lora_target: all

### Dataset
dataset: train_data_1k
dataset_dir: data
template: qwen
cutoff_len: 512

### Output
output_dir: lora_r16_1k_model
overwrite_output_dir: true

### LoRA
lora_rank: 16
lora_alpha: 512
lora_dropout: 0.05

### Training Hyperparameters
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 3.0e-4
num_train_epochs: 3
lr_scheduler_type: cosine
warmup_ratio: 0.05

### Evaluation
val_size: 0.1
per_device_eval_batch_size: 4
eval_strategy: steps
eval_steps: 100

### Save
save_steps: 500
save_total_limit: 2

### Other
bf16: true
logging_steps: 50
"""

# Save config to a YAML file
with open('lora_r16_1k.yaml', 'w') as f:
    f.write(config_r16_1k)

print("Training config saved to lora_r16_1k.yaml")
print("\nKey LoRA settings:")
print("  lora_rank    : 16")
print("  lora_alpha   : 512  (scaling = alpha/rank = 32)")
print("  lora_dropout : 0.05")
print("  lora_target  : all")
print("\nKey training settings:")
print("  Effective batch size : 16  (batch_size=4 × grad_accum=4)")
print("  Learning rate        : 3e-4")
print("  Epochs               : 3")
print("  Cutoff length        : 512 tokens")
print("  Template             : qwen")

## Step 6 — Train LoRA Rank=16 Model

We launch training using LLamaFactory's CLI. The command reads the YAML config and handles everything — model loading, LoRA adapter setup, training loop, validation, and saving the final adapter weights.

**What gets saved after training?**  
LLamaFactory saves the trained LoRA adapter to the `output_dir` specified in the config (`lora_r16_1k_model/`). This folder contains:
- `adapter_config.json` — LoRA configuration (rank, alpha, target modules, etc.)
- `adapter_model.safetensors` — the actual trained adapter weights
- tokenizer files — needed to reload the model later

Note: The base model weights are **not** saved — only the small adapter. This is the key efficiency advantage of LoRA.

In [ ]:
# Launch LoRA training via LLamaFactory CLI
# This reads lora_r16_1k.yaml and runs the full training loop
!llamafactory-cli train lora_r16_1k.yaml

### Verify Training Output
We check that the adapter files were saved correctly before attempting to load the model.

In [ ]:
import os

output_dir = 'lora_r16_1k_model'

if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    print(f"   Output directory exists: {output_dir}/")
    print(f"   Files saved: {files}")

    if 'adapter_config.json' in files:
        print("   adapter_config.json found!")
    if 'adapter_model.safetensors' in files or 'adapter_model.bin' in files:
        print("   adapter weights found!")
    else:
        print("   WARNING: No adapter weight file found — training may have failed")
else:
    print(f"   ERROR: Output directory '{output_dir}' not found — training failed")

## Step 7 — Load the Fine-Tuned Model

To use the trained model for inference, we load it in two steps:
1. Load the **base model** (Qwen2.5-7B) with frozen weights
2. Wrap it with the **LoRA adapters** using `PeftModel.from_pretrained`

**What is PeftModel?**  
PEFT (Parameter-Efficient Fine-Tuning) is HuggingFace's library for loading LoRA adapters on top of a base model. `PeftModel.from_pretrained` reads the `adapter_config.json` and `adapter_model.safetensors` files saved during training, and injects the adapter weights into the correct layers of the base model.

**Why `padding_side = 'left'`?**  
For batched generation with causal language models, sequences of different lengths must be padded to the same length. Left-padding ensures all sequences end at the same position, so the model generates new tokens correctly for each sequence in the batch.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

MODEL_NAME = "Qwen/Qwen2.5-7B"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,  # bfloat16 is recommended for Qwen2.5
    device_map="auto"            # automatically place on GPU
)

print("Loading LoRA adapters from lora_r16_1k_model/...")
# PeftModel wraps the base model and injects the trained LoRA adapter weights
finetuned_model = PeftModel.from_pretrained(
    base_model,
    "lora_r16_1k_model",          # folder containing adapter_config.json + adapter weights
    torch_dtype=torch.bfloat16
)

# Set model to evaluation mode — disables dropout
finetuned_model.eval()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Qwen2.5 has a built-in padding token — no manual assignment needed
# Left-padding is required for batched generation with causal language models
tokenizer.padding_side = "left"

print(f"\n   Fine-tuned model loaded successfully!")
print(f"   Device         : {next(finetuned_model.parameters()).device}")

# Count trainable (LoRA) vs total parameters
trainable_params = sum(p.numel() for p in finetuned_model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in finetuned_model.parameters())
print(f"   LoRA parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}% of total)")
print(f"   Total parameters: {total_params:,}")

## Step 8 — Test on a Single Example
Before running the full evaluation, we test on one example to verify the fine-tuned model generates output correctly.

**How does text generation work here?**
1. We build a prompt: `"Summarize the following article:\n\n{article}\n\nSummary:"`
2. The tokenizer converts this prompt into token IDs
3. The model generates new tokens after `"Summary:"`
4. The tokenizer decodes the output tokens back into text
5. We extract only the part after `"Summary:"` as the generated summary

**What is `torch.no_grad()`?**  
During inference (not training), we don't need to compute gradients. `torch.no_grad()` disables gradient tracking, saving memory and speeding up inference significantly.

In [ ]:
from datasets import load_dataset
import time

# Load the test set (we need it for this single test and the full evaluation later)
print("Loading CNN/DailyMail test set (1,000 examples)...")
test_dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")
test_dataset_1k = test_dataset.select(range(1000))
print(f"   Loaded {len(test_dataset_1k)} test examples")

# Pick the first test example
test_example = test_dataset_1k[0]

# Build the same prompt format used during training
prompt = f"Summarize the following article:\n\n{test_example['article']}\n\nSummary:"

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)
inputs = {k: v.to(finetuned_model.device) for k, v in inputs.items()}

start_time = time.time()

# Generate summary — no gradients needed during inference
with torch.no_grad():
    outputs = finetuned_model.generate(
        **inputs,
        max_new_tokens=100,                    # generate at most 100 new tokens
        do_sample=False,                       # greedy decoding
        pad_token_id=tokenizer.pad_token_id    # Qwen2.5 has its own pad token
    )

elapsed = time.time() - start_time

# Decode and extract the summary portion
input_length      = inputs['input_ids'].shape[1]
generated_tokens  = outputs[0][input_length:]
generated_summary = tokenizer.decode(generated_tokens, skip_special_tokens=True)[:200]

print("\n" + "=" * 60)
print("SINGLE EXAMPLE TEST")
print("=" * 60)
print(f"Article (first 200 chars) : {test_example['article'][:200]}")
print(f"\nReference Summary         : {test_example['highlights']}")
print(f"\nGenerated Summary         : {generated_summary}")
print(f"\nGeneration Time           : {elapsed:.2f} seconds")
print("\n   Single example test passed! Proceeding to full evaluation...")

## Step 9 — Evaluate on 1,000 Test Examples

We run the fine-tuned model on all 1,000 test examples using **batched evaluation** and collect generated summaries.

**Why batched evaluation?**  
Batched evaluation processes multiple examples simultaneously, making much better use of GPU resources and reducing total evaluation time compared to sequential processing.

**Adjusting `BATCH_SIZE`:**  
The batch size controls how many examples are processed at once. Reduce it if you get an out-of-memory error:
- 40GB+ VRAM: `BATCH_SIZE = 8`
- 24GB VRAM: `BATCH_SIZE = 4`
- 16GB VRAM: `BATCH_SIZE = 2`

In [ ]:
from tqdm import tqdm
import time

# BATCH_SIZE controls how many examples are processed simultaneously
# Qwen2.5-7B is a large model (~15GB in bfloat16) — adjust based on your GPU VRAM
# Recommended starting points:
#   - 40GB+ VRAM: BATCH_SIZE = 8
#   - 24GB VRAM:  BATCH_SIZE = 4
#   - 16GB VRAM:  BATCH_SIZE = 2
#   - If you get an out-of-memory error, reduce BATCH_SIZE
BATCH_SIZE = 4

all_summaries  = []
all_references = []

total_examples = len(test_dataset_1k)
num_batches = (total_examples + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Starting batched evaluation on {total_examples:,} test examples...")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Total batches : {num_batches}")
print("=" * 60)

start_total = time.time()

for batch_idx in tqdm(range(num_batches), desc="Evaluating batches"):

    start_idx = batch_idx * BATCH_SIZE
    end_idx   = min(start_idx + BATCH_SIZE, total_examples)

    batch = test_dataset_1k.select(range(start_idx, end_idx))

    prompts = [
        f"Summarize the following article:\n\n{example['article']}\n\nSummary:"
        for example in batch
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(finetuned_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = finetuned_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],  # tells model which tokens are real vs padding
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    for output in outputs:
        input_length      = inputs['input_ids'].shape[1]
        generated_tokens  = output[input_length:]
        generated_summary = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        all_summaries.append(generated_summary)

    all_references.extend(batch["highlights"])

    examples_done = end_idx
    if examples_done % 200 == 0 or examples_done == total_examples:
        elapsed = time.time() - start_total
        print(f"   [{examples_done}/{total_examples}] completed | Time elapsed: {elapsed/60:.1f} mins")

total_time = time.time() - start_total
print(f"\n   Evaluation complete!")
print(f"   Total examples evaluated : {len(all_summaries)}")
print(f"   Total time               : {total_time/60:.1f} minutes")

## Step 10 — Calculate BLEU and ROUGE Scores

We compare the generated summaries against the reference summaries using four metrics:

| Metric | What it measures |
|--------|------------------|
| **BLEU-4** | Overlap of 4-word phrases between generated and reference summary |
| **ROUGE-1** | Overlap of individual words (unigrams) |
| **ROUGE-2** | Overlap of 2-word phrases (bigrams) |
| **ROUGE-L** | Longest common subsequence between generated and reference |

In [ ]:
from evaluate import load

print("Calculating BLEU and ROUGE scores...")
print("=" * 60)

bleu_metric  = load("bleu")
rouge_metric = load("rouge")

bleu_result = bleu_metric.compute(
    predictions=all_summaries,
    references=[[r] for r in all_references]
)

rouge_result = rouge_metric.compute(
    predictions=all_summaries,
    references=all_references
)

# Extract and scale scores to 0-100
# BLEU-4 = index 3 of precisions (4-gram precision)
bleu4  = bleu_result['precisions'][3] * 100
rouge1 = rouge_result['rouge1'] * 100
rouge2 = rouge_result['rouge2'] * 100
rougeL = rouge_result['rougeL'] * 100

print(f"\n{'='*60}")
print(f"  LoRA r=16 RESULTS — 1k Training | 1,000 Test Examples")
print(f"{'='*60}")
print(f"  BLEU-4   : {bleu4:.2f}")
print(f"  ROUGE-1  : {rouge1:.2f}")
print(f"  ROUGE-2  : {rouge2:.2f}")
print(f"  ROUGE-L  : {rougeL:.2f}")
print(f"{'='*60}")

## Step 11 — Save Results
We save the scores to a JSON file in the current directory. Download it from the Colab file browser (left sidebar → right-click → Download).

In [ ]:
import json

lora_r16_1k_results = {
    "experiment"     : "LoRA rank=16 — Qwen2.5-7B",
    "model"          : "Qwen/Qwen2.5-7B",
    "lora_rank"      : 16,
    "lora_alpha"     : 512,
    "train_samples"  : 900,
    "test_samples"   : len(all_summaries),
    "bleu_4"         : round(bleu4, 4),
    "rouge_1"        : round(rouge1, 4),
    "rouge_2"        : round(rouge2, 4),
    "rouge_L"        : round(rougeL, 4)
}

# Save to current directory (Colab's /content/ folder)
# To download: right-click the file in the Colab file browser (left sidebar) and select Download
with open("lora_r16_1k_results.json", "w") as f:
    json.dump(lora_r16_1k_results, f, indent=2)

print("   Results saved to lora_r16_1k_results.json")
print(json.dumps(lora_r16_1k_results, indent=2))